In [1]:
import pandas as pd

order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
orders = pd.read_csv("../data/olist_orders_dataset.csv")

order_items.head()


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [2]:
order_summary = (
    order_items
    .groupby("order_id")
    .agg(
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        item_count=("order_item_id", "count"),
        avg_item_price=("price", "mean")
    )
    .reset_index()
    .sort_values("total_price", ascending=False)
)

order_summary.head(10)

,order_id,total_price,total_freight,item_count,avg_item_price
1455,03caa2c082116e1d31e67e9ae3700499,13440.0,224.08,8,1680.00
44467,736e1922ae60d0d6a89247b851902527,7160.0,114.88,4,1790.00
3130,0812eb902a67711a1cb742b3cdaa65ae,6735.0,194.31,1,6735.00
98298,fefacc66af859508bf1a7934eab1e97f,6729.0,193.21,1,6729.00
94439,f5136e38d1a14a4dbd87dff67da82701,6499.0,227.66,1,6499.00
17114,2cc9089445046817a7539d90805e6e5a,5934.6,146.94,6,989.10
65046,a96610ab360d42a2e5335a3998b4718a,4799.0,151.34,1,4799.00
9868,199af31afc78c699f0dbf71fb178d4d4,4690.0,74.34,1,4690.00
69531,b4c4b76c642808cbe472a32b86cddc95,4599.9,209.54,2,2299.95
54353,8dbc85d1447242f3b127dda390d56e19,4590.0,91.78,1,4590.00


In [3]:
orders_with_totals = orders.merge(
    order_summary,
    on="order_id",
    how="left"
)
orders_with_totals.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,total_price,total_freight,item_count,avg_item_price
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,29.99,8.72,1.0,29.99
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,118.70,22.76,1.0,118.70
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,159.90,19.22,1.0,159.90
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,45.00,27.20,1.0,45.00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,19.90,8.72,1.0,19.90


In [4]:
print("Before merge:", len(orders))
print("After merge:", len(orders_with_totals))

Before merge: 99441
After merge: 99441


In [5]:
orders_with_totals[
    ["total_price", "total_freight", "item_count"]
].isna().sum()

total_price      775
total_freight    775
item_count       775
dtype: int64

In [6]:
orders_clean = orders_with_totals.copy()

orders_clean["item_count"] = (
    orders_clean["item_count"].fillna(0).astype(int)
)

In [7]:
print(orders_clean["item_count"].isna().sum())
print(orders_clean["item_count"].dtype)

0
int64


In [8]:
orders_clean.duplicated(subset=["order_id"]).sum()

np.int64(0)

In [9]:
duplicate_orders = orders_clean.duplicated(
    subset=["order_id"]
).sum()

print("Duplicate order IDs:", duplicate_orders)

Duplicate order IDs: 0


In [10]:
orders_clean["order_purchase_timestamp"].dtype

<StringDtype(storage='python', na_value=nan)>

In [11]:
orders_clean["order_purchase_timestamp"] = pd.to_datetime(
    orders_clean["order_purchase_timestamp"],
    errors="coerce"
)

In [12]:
print(orders_clean["order_purchase_timestamp"].dtype)
print(orders_clean["order_purchase_timestamp"].isna().sum())

datetime64[us]
0


In [13]:
result = (
    orders_clean
    .groupby("order_status")
    .agg(
    order_count = ("order_id","count"),
    total_revenue = ("total_price","sum"),
    avg_order_value = ("total_price","mean")
)
.reset_index()
.sort_values("total_revenue", ascending=False)
)
result

,order_status,order_count,total_revenue,avg_order_value
3,delivered,96478,13221498.11,137.041586
6,shipped,1107,150727.44,136.281591
1,canceled,625,95235.27,206.584100
4,invoiced,314,61526.37,197.199904
5,processing,301,60439.22,200.794751
7,unavailable,609,2007.69,334.615000
0,approved,2,209.60,104.800000
2,created,5,0.00,NaN


In [14]:
orders_clean["month"] = (
    orders_clean["order_purchase_timestamp"]
    .dt.to_period("M")
)

monthly_summary = (
    orders_clean
    .groupby("month")
    .agg(
        order_count=("order_id", "count"),
        total_revenue=("total_price", "sum"),
        avg_order_value=("total_price", "mean")
    )
    .reset_index()
    .sort_values("month")
)

monthly_summary

,month,order_count,total_revenue,avg_order_value
0,2016-09,4,267.36,89.120000
1,2016-10,324,49507.66,160.739156
2,2016-12,1,10.90,10.900000
3,2017-01,800,120312.87,152.487795
4,2017-02,1780,247303.02,142.702262
5,2017-03,2682,374344.30,141.743393
6,2017-04,2404,359927.23,150.534182
7,2017-05,3700,506071.14,138.270803
8,2017-06,3245,433038.60,134.609450
9,2017-07,4026,498031.48,125.480343


In [17]:
res = pd.DataFrame([{
    "order_count": order_items["order_id"].nunique(),
    "item_count": len(order_items),
    "total_price": order_items["price"].sum()
}])

res

,order_count,item_count,total_price
0,98666,112650,13591643.7


In [18]:
order_count = order_items["order_id"].nunique()
item_count = len(order_items)
total_price = order_items["price"].sum()

print("Order count:", order_count)
print("Item count:", item_count)
print("Total price:", total_price)

Order count: 98666
Item count: 112650
Total price: 13591643.7


In [22]:
product_stats = ( 
    order_items.groupby('product_id', as_index = False)
    .agg(
        items_sold = ("price", "count"),
        total_revenue = ("price", "sum"),
        avg_price = ("price", "mean")
    )
    .sort_values(by="total_revenue", ascending=False)
    .head(10)
)

print(product_stats)

                             product_id  items_sold  total_revenue  \
24086  bb50f2e236e5eea0100680137654686c         195       63885.00   
14068  6cdd53843498f92890544667809f1595         156       54730.20   
27613  d6160fb7873f184099d9bc95e30376af          35       48899.34   
27039  d1c427060a0f73f6b889a5c7c61f2ac4         343       47214.51   
19742  99a4788cb24856965c36a24e339b6058         488       43025.56   
8051   3dd2a17168ec895c781a9191c1e95ad7         274       41082.60   
4996   25c38557cf793876c5abdd5931f922db          38       38907.32   
12351  5f504b3a1c75b73d6151be81eb05bdc9          63       37733.90   
10867  53b36df67ebb7c41585e8d54d6772e08         323       37683.42   
22112  aca2eb7d00ea1a7b8ebd4e68314663af         527       37608.90   

         avg_price  
24086   327.615385  
14068   350.834615  
27613  1397.124000  
27039   137.651633  
19742    88.167131  
8051    149.936496  
4996   1023.876842  
12351   598.950794  
10867   116.666935  
22112    71.364137 

In [ ]:
order_counts = (
    order_items.groupby('order_id', as_index=False)
    .agg(item_count=('order_item_id', 'count'))
)

orders_merged = orders_clean.merge(order_counts, on='order_id', how='left')
orders_merged['item_count'] = orders_merged['item_count'].fillna(0).astype(int)

orders_without_items = orders_merged[orders_merged['item_count'] == 0]

print(len(orders_without_items))

In [23]:
total_rows = len(order_items)
unique_orders = order_items['order_id'].nunique()
total_price = round(order_items['price'].sum(), 2)

print(total_rows, unique_orders, total_price)

112650 98666 13591643.7
